In [ ]:
import sys
sys.path.append('..')

from config.settings import Settings
from features.embeddings import load_gensim_embeddings
from features.vectorizer import TextVectorizerModel
from datasets.loader import load_splits
from datasets.paths import ProjectPaths
from gensim.models import KeyedVectors
import string
import unicodedata


c:\Users\malos\Documents\GitHub\JustShare\server\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [ ]:
settings = Settings()
print(settings)


batch_size=64 mlp_dropout=0.4 lstm_dropout=0.3 pooling='mean' similarity='cosine' hidden_dim=64 bidirectional=False mlp_layers=[32] concat_features=['diff'] epochs=20 augmented_data=False siamese_name='lstm_mean_cosine' max_len=26 augmented_max_len=26 case=False strip_punctuation=True


In [ ]:
paths = ProjectPaths(siamese_name=settings.siamese_name)


In [ ]:
if settings.augmented_data:
	max_len = settings.augmented_max_len
	train_dir = paths.augmented_dir
else:
	max_len = settings.max_len
	train_dir = paths.processed_dir

print(max_len)

26


In [ ]:
splits = {
	"train": train_dir,
	"dev": paths.processed_dir,
    "test": paths.processed_dir
}

datasets = load_splits(splits)

train_df = datasets["train"]
dev_df = datasets["dev"]
test_df = datasets["test"]


In [ ]:
display(train_df)


,sentence1,sentence2,score,split,score_norm
0,Un avión está despegando.,Un avión está despegando.,5.00,train,1.00
1,Un hombre está tocando una gran flauta.,Un hombre está tocando una flauta.,3.80,train,0.76
2,Un hombre está untando queso rallado en una pi...,Un hombre está untando queso rallado en una pi...,3.80,train,0.76
3,Tres hombres están jugando al ajedrez.,Dos hombres están jugando al ajedrez.,2.60,train,0.52
4,Un hombre está tocando el violonchelo.,Un hombre sentado está tocando el violonchelo.,4.25,train,0.85
...,...,...,...,...,...
5736,Vendavales severos mientras la tormenta Clodag...,Merkel promete la solidaridad de la OTAN con L...,0.00,train,0.00
5737,Docenas de egipcios rehenes tomados por terror...,El número de muertos en el accidente de un bar...,0.00,train,0.00
5738,El Presidente se dirige a Bahrein,Presidente Xi: China seguirá ayudando a combat...,0.00,train,0.00
5739,China y la India se comprometen a fomentar los...,China lucha por tranquilizar a los nerviosos c...,0.00,train,0.00


In [ ]:
print("Train length:", len(train_df))
print("Dev length:", len(dev_df))


Train length: 5741
Dev length: 1497


In [ ]:
wv = KeyedVectors.load_word2vec_format(paths.word2vec_path, binary=True)


In [ ]:
dim = wv.vector_size
print(dim)
print(len(wv))


400
1943871


In [ ]:
words = list(wv.key_to_index)

upper_words = [w for w in words if any(c.isupper() for c in w)]
lower_words = [w for w in words if w.islower()]
other_words = [
    w for w in words
    if not any(c.isupper() for c in w) and not w.islower()
]

total = len(words)

print(f"Total palabras: {total}")
print(f"Con mayúsculas: {len(upper_words)} ({len(upper_words)/total*100:.2f}%)")
print(f"Solo minúsculas: {len(lower_words)} ({len(lower_words)/total*100:.2f}%)")
print(f"Otros: {len(other_words)} ({len(other_words)/total*100:.2f}%)")

print("\nEjemplos de 'otros':")
print(other_words[:100])


Total palabras: 1943871
Con mayúsculas: 9817 (0.51%)
Solo minúsculas: 1903810 (97.94%)
Otros: 30244 (1.56%)

Ejemplos de 'otros':
['—', '1', '–', '2', '3', '4', '5', '6', '10', '7', '12', '8', '9', '11', '15', '20', '13', '14', '30', '16', '18', '25', '17', '22', '19', '24', '23', '21', '26', '27', '28', '100', '•', '29', '31', '50', '·', '2000', '©', '…', '40', '1936', '1945', '45', '60', '2001', '1999', '1939', '1944', '1995', '1940', '35', '1998', '1990', '¡', '1980', '1941', '32', '¿', '1942', '38', '1989', '2002', '1997', '1996', '36', '2003', '2009', '1986', '34', '1943', '1985', '1937', '1992', '1970', '1994', '2004', '2010', '33', '1976', '1977', '1987', '1991', '200', '2005', '1979', '1984', '1993', '1982', '1938', '1975', '1981', '1968', '2006', '2011', '70', '1988', '1978', '2008', '1973']


In [ ]:
for p in string.punctuation:
    print(p, p in wv.key_to_index)
    

! False
" False
# False
$ False
% False
& False
' False
( False
) False
* False
+ False
, False
- False
. False
/ False
: False
; False
< False
= False
> False
? False
@ False
[ False
\ False
] False
^ False
_ False
` False
{ False
| False
} False
~ False


In [ ]:
punct_tokens = [
    w for w in wv.key_to_index
    if all(unicodedata.category(c).startswith(("P", "S")) for c in w)
]

print(punct_tokens)

['—', '–', '•', '·', '©', '…', '¡', '¿', '«', '»', '—…', '―', '—¡', '—¿', '€', '«…', '−', '×', '“', '─', '‹', '˜', '”', '»—', '↵', '§', '—«', '°', '«¡', '›', '●', '…»', '———', '†', '→', '®', '——', '£', '—¿…', '—«…', '’', '‹›', '¿…', '«»', '«¿', '—”', '»…', '¡…', '………………………', '∗', '—¡…', '“…', '±', '÷', '«—', '‘', '§§', '―»', '´', '–…', '„', '‹‹', '¥', '—————————————————', '∞', '℃', '™', '¶', '‡', '✤', '□', '■', '¨', '¡¡', '‑', '—›', '»¿', '…”', '⇒', '—•', '¡»', '»¡', '“—', '—»', '“¡', '¦', '—————', '……………', '√', '≠', '¿¿', '—“', '”»', '‚', '€¨‘', '–¿', '♥', '∏', '¡¿', '¯', '————————', '➔', '¬', '¤', '––', '«€»', '“”', '¿¿¿', '≈', '«¡…', '¢', '…………', '≤', '————————————', '“¿', '«¿…', '››', '—«¡', '¸', '––––––––––––––––––––', '–¡', '••', '↑', '···', '……', '⋅', '—¡¡', '♣', '↑〉', '——————————————————————————————————————————————', '………', '————', '•—', '††', '¡—', '•••', '¡¡¡', '——————', '«¿»', '★', '↔', '↓〉', '££', '»—¿', '……………………', '¿¡', '———————', '—«€»', '↗〉', '´´', '…………………', '—————————

In [ ]:
for signo in [".", ",", ";", ":", "!", "?", ")", '"', "'"]:
    n = sum(token.endswith(signo) for token in wv.key_to_index)
    print(signo, n)


. 0
, 0
; 0
: 0
! 0
? 0
) 0
" 0
' 0


In [ ]:
all_sentences = list(train_df["sentence1"]) + list(train_df["sentence2"])

vectorizer = TextVectorizerModel(
    max_len=max_len,
    case=settings.case,
    strip_punctuation=settings.strip_punctuation
)

vectorizer.adapt(all_sentences)

vocab = vectorizer.get_vocabulary()
word2idx = {word: idx for idx, word in enumerate(vocab)}
print(f"Vocabulary size: {len(vocab)}")


Vocabulary size: 13613


In [ ]:
vocab = vectorizer.vectorizer.get_vocabulary()
print(vocab[:10])


['', '[UNK]', np.str_('de'), np.str_('la'), np.str_('el'), np.str_('en'), np.str_('un'), np.str_('una'), np.str_('a'), np.str_('los')]


In [ ]:
embedding_matrix, missing_words = load_gensim_embeddings(wv, word2idx, dim)

print(embedding_matrix.shape)

print(missing_words[:50])


Encontradas: 13226/13611 (97.17%)
No encontradas: 385/13611 (2.83%)
(13613, 400)
[np.str_('ixic'), np.str_('neener'), np.str_('us30ytrr'), np.str_('Äôs'), np.str_('us10ytrr'), np.str_('promorsi'), np.str_('lendingtree'), np.str_('sorenstam'), np.str_('sistánbaluchistán'), np.str_('mh17'), np.str_('inglésturco'), np.str_('hakimullah'), np.str_('152015'), np.str_('strier'), np.str_('someoen'), np.str_('shalgam'), np.str_('saferworld'), np.str_('prorusia'), np.str_('pacÍficos'), np.str_('xanis'), np.str_('x86'), np.str_('wolfcale'), np.str_('weisselberg'), np.str_('waksal'), np.str_('w32sobigcmm'), np.str_('usvisit'), np.str_('torsella'), np.str_('tafb'), np.str_('syndia'), np.str_('studabaker'), np.str_('stonesoft'), np.str_('staffenberg'), np.str_('sriyanto'), np.str_('solinvictus'), np.str_('smeone'), np.str_('shereka'), np.str_('seemandhra'), np.str_('rs24'), np.str_('ropeik'), np.str_('romeril'), np.str_('reutersipsos'), np.str_('raghdad'), np.str_('promursi'), np.str_('pribbenow'), 